# Quickstart: data and the model interface

TorchPP models events in continuous time: each event has a timestamp and an *event type*
(also called a *mark*). New to point processes? Start with
[`00_tpp_introduction.ipynb`](./00_tpp_introduction.ipynb).

This notebook covers the two things you need to get started:

1. the **data structures**: `MVEventData` (one sequence) and `BatchedMVEventData` (a padded
   batch), and
2. the **model interface**: the probabilistic API every model exposes, shown on the simplest
   model, the Poisson process.

In [ ]:
import matplotlib.pyplot as plt
import torch

from torchpp import BatchedMVEventData, MVEventData, PoissonProcess

## The data interface

Event data lives in two classes: `MVEventData` holds a single sequence, and its
batched, right-padded counterpart is `BatchedMVEventData`. Both store their event
times and marks in a `tensordict.TensorDict` and share a common interface from the
abstract base `EventSequence` — so a batch behaves like a tensor container (it
slices and moves between devices like one).

In [ ]:
# A sequence is event times + integer event types in [0, D), and an optional observation
# window T_obs. If T_obs is omitted it defaults to the last event time.
seqs = [
    MVEventData(torch.tensor([0.10, 0.34, 0.57, 0.91]), torch.tensor([0, 1, 0, 2]), T_obs=1.0),
    MVEventData(torch.tensor([0.05, 0.22, 0.40]), torch.tensor([2, 1, 0]), T_obs=2.0),
    MVEventData(torch.tensor([0.8, 2.3, 4.0, 7.9, 17.2]), torch.tensor([2, 1, 0, 2, 1])),
]
print("seq 0 times:", seqs[0].time_points.tolist())
print("seq 0 types:", seqs[0].event_types.tolist())

A sequence on its time axis. Each row is one sequence, colours are event types, and the
bar marks the end of the observation window `T_obs`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.2))
for row, s in enumerate(seqs):
    ax.scatter(s.time_points, [row] * len(s), c=[f"C{m}" for m in s.event_types.tolist()], s=40, zorder=3)
    T = s.T_obs if s.T_obs is not None else float(s.time_points[-1])
    ax.plot([0, T], [row, row], color="0.7", lw=1)
    ax.plot([T, T], [row - 0.25, row + 0.25], color="k", lw=1.5)
ax.set_yticks(range(len(seqs)), [f"seq {i}" for i in range(len(seqs))])
ax.set_xlabel("time"); ax.set_ylim(-0.6, len(seqs) - 0.4)
plt.show()

In [ ]:
# Build a batch two equivalent ways: the constructor, or the from_sequences classmethod.
batch = BatchedMVEventData(mv_events=seqs)
batch_b = BatchedMVEventData.from_sequences(seqs)
assert torch.equal(batch.time_points, batch_b.time_points)
assert batch.time_points.shape == (3, 5)  # (B, max_len); shorter sequences are right-padded

# Things you can read off a batch. Time-valued queries take one time per sequence, (B,):
t_query = torch.full((3,), 0.7)
print("valid-event mask:         ", batch.event_mask.tolist())  # (B, L) real vs padding
print("observation window T_obs: ", batch.T_obs.tolist())  # (B,)
print("last event time:          ", batch.last_event_time.tolist())  # (B,)
print("events before t=0.7:      ", batch.count_events_before(t_query).tolist())  # (B,)
print("last event before t=0.7:  ", batch.event_time_before(t_query).tolist())  # (B,)

## The model interface

Every model inherits from `TemporalPointProcess`:

```
TemporalPointProcess          the family-agnostic contract
└── AutoregressiveTPP          likelihood factorizes over events
    ├── IntensityBasedTPP      defined by an intensity λ(t)
    └── DensityBasedTPP        defined by an inter-event density
```

`AutoregressiveTPP` fixes one **uniform probabilistic interface**. Given a
history `H`, a model describes the **next** event — a pair
(time `t`, event type `m`) — and you can query that distribution every way. The calls
are the same whether the model is intensity- or density-based.

*Joint, marginals, and conditionals of the next event (t, m):*

| method | shape | meaning |
|---|---|---|
| `density_joint(t, ts)` | (B, [Q,] D) | p(t, m) — the joint next-event density |
| `density_time(t, ts)` | (B, [Q]) | p(t) = Σ_m p(t, m) — *when* the next event is |
| `prob_mark(ts)` | (B, D) | p(m) — *which* mark, marginal over time |
| `prob_mark_given_time(t, ts)` | (B, [Q,] D) | p(m \| t) — the mark, given the time |
| `density_time_given_mark(t, ts)` | (B, [Q,] D) | p(t \| m) — the time, given the mark |

*Survival/CDF of the next-event time:*

| method | shape | meaning |
|---|---|---|
| `survival_time(t, ts)` | (B, [Q]) | S(t) = P(next event ≥ t) |
| `cdf_time(t, ts)` | (B, [Q]) | P(next event < t) = 1 − S(t) |
| `time_mean(ts)` | (B,) | E[t] — expected next-event time |
| `time_quantile(u, ts)` | (B,) | inverse CDF (median at u = 0.5) |

*Training and generation:*

| method | meaning |
|---|---|
| `likelihood(ts)` | log-likelihood of the observed events, often the training objective |
| `sample(ts, num_steps)` | roll the process forward from a history |

Conventions: the time methods accept `t` of shape (B,) or (B, Q), return
log-values by default (pass `log=False` for the plain value), and describe the
*next* event after the observation window — so `t` must be at or after each
sequence's `T_obs` (its last event time when `T_obs` is not given); otherwise
they raise an error. A `T_obs` after the last event means no event was observed
in between, and the predictions condition on that. Cut a sequence to query
other times.

Below we use the **Poisson process**: the simplest TPP, with a constant intensity
(so inter-event times are exponentially distributed). It is an
`IntensityBasedTPP`.

In [ ]:
model = PoissonProcess(D=3, terminating=False)  # 3 event types; constant intensity

# The model is UNTRAINED (random rates): we are showing the *interface*, not fitted
# values (notebook 02 fits a model). Predictive queries describe the NEXT event after
# the observation window, so the query time must be at or after each sequence's T_obs
# (the last event time when T_obs is not given).
t = batch.T_obs + 0.05  # (B,) one "next event at t" query per sequence

with torch.no_grad():
    # Training objective: log-likelihood of the observed events. (B,)
    print("log-likelihood:         ", [round(v, 3) for v in model.likelihood(batch).tolist()])

    # The next-event distribution over (time t, mark m), queried every way:
    print("p(t, m)  density_joint: ", model.density_joint(t, batch, log=False))  # (B, D)
    print("p(t)     density_time:  ", [round(v, 3) for v in model.density_time(t, batch, log=False).tolist()])
    print("p(m)     prob_mark s0:  ", [round(v, 3) for v in model.prob_mark(batch, log=False)[0].tolist()])
    print(
        "p(m|t)   given time s0: ", [round(v, 3) for v in model.prob_mark_given_time(t, batch, log=False)[0].tolist()]
    )
    print("p(t|m)   given mark:    ", model.density_time_given_mark(t, batch, log=False))  # (B, D)
    # For a Poisson the mark is independent of the time, so p(m|t) == p(m) above.

    # Survival / CDF / summaries of the next-event time:
    print("S(t) = P(next >= t):    ", [round(v, 3) for v in model.survival_time(t, batch, log=False).tolist()])
    print("F(t) = P(next <  t):    ", [round(v, 3) for v in model.cdf_time(t, batch, log=False).tolist()])
    print("E[t_next]:              ", [round(v, 3) for v in model.time_mean(batch).tolist()])
    print("median next-event time: ", [round(v, 3) for v in model.time_quantile(0.5, batch).tolist()])

    # Generation: roll the process forward from one history.
    sample = model.sample(seqs[0], num_steps=5)   # a Sample(sequence, times, mark_probs)
print("sampled event times (absolute):", [round(float(x), 3) for x in sample.times])
print("history + samples:", [round(x, 2) for x in sample.sequence.time_points.tolist()])

## The intensity view

The interface above is family-agnostic — it is what *every* model answers. An
`IntensityBasedTPP` realizes it through two intensity-specific functions:

| method | shape | meaning |
|---|---|---|
| `intensity(t, ts)` | (B, D) or (B, Q, D) | λ_d(t) — the instantaneous rate (hazard) of a type-d event at t |
| `cumulative_intensity(T, ts)` | (B, D) | Λ_d(T) = ∫ λ_d — the integrated rate (the "compensator") |

Everything probabilistic is built from these two:

  p(t, m | H) = λ_m(t) · S(t),   with   S(t) = exp( −Σ_d ∫ λ_d ).

For the **Poisson** process the intensity is constant in time, λ_d(t) = μ_d, so
the inter-event time is exponential and the mark is independent of the time
(p(m | t) = p(m) = μ_d / Σ_d μ_d).

In [ ]:
with torch.no_grad():
    # lambda_d(t) is constant for a Poisson — the same at any query time. (B, D)
    print("intensity λ(t=0.5) s0:", [round(v, 3) for v in model.intensity(torch.full((3,), 0.5), batch)[0].tolist()])
    print("intensity λ(t=5.0) s0:", [round(v, 3) for v in model.intensity(torch.full((3,), 5.0), batch)[0].tolist()])

    # Cumulative intensity Λ_d(T) = ∫ λ_d — the integrated rate. (B, D)
    T = batch.last_event_time + 1.0
    print("cumulative intensity Λ(T) s0:", [round(v, 3) for v in model.cumulative_intensity(T, batch)[0].tolist()])

    # Everything probabilistic is built from λ and S. Check p(t, m) = λ_m(t) · S(t):
    lam = model.intensity(t, batch)  # (B, D)
    S = model.survival_time(t, batch, log=False)  # (B,)
    diff = (lam * S.unsqueeze(-1) - model.density_joint(t, batch, log=False)).abs().max()
    print("max |λ_m(t)·S(t) − density_joint(t)|:", float(diff))

## Optional: covariates

A sequence can carry static covariates as a dict of 1-D tensors. The built-in models do not use
them; your own models can (see the survival notebooks).

In [ ]:
cov_seqs = [
    MVEventData(
        torch.tensor([0.1, 0.2, 0.3]),
        torch.tensor([0, 1, 2]),
        T_obs=1.0,
        covariates={"age": torch.tensor([55.0]), "sex": torch.tensor([1.0])},
    ),
    MVEventData(
        torch.tensor([0.5, 0.6]),
        torch.tensor([0, 1]),
        T_obs=1.0,
        covariates={"age": torch.tensor([27.2]), "sex": torch.tensor([0.0])},
    ),
]
print("seq 0 covariates:", cov_seqs[0].covariates)

# Covariates stack across the batch to (B, feature_dim); every sequence must share keys.
cov_batch = BatchedMVEventData.from_sequences(cov_seqs)
print("batched covariates:", cov_batch.covariates)